In [ ]:
!pip install "pandas<2.2.0" wfdb

In [ ]:
#librerias
import wfdb
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
from scipy.signal import medfilt

# Lista oficial de registros de la base de datos MIT-BIH
registros = [
    '100', '101', '102', '103', '104', '105', '106', '107', '108', '109',
    '111', '112', '113', '114', '115', '116', '117', '118', '119', '121',
    '122', '123', '124', '200', '201', '202', '203', '205', '207', '208',
    '209', '210', '212', '213', '214', '215', '217', '219', '220', '221',
    '222', '223', '228', '230', '231', '232', '233', '234'
]

In [ ]:

# Bucle para recorrer cada número de registro
for num in registros[:1]:
    try:
        print(f"Procesando registro: {num}...")

        # Leer el registro (.dat y .hea)
        record = wfdb.rdrecord(num, pn_dir='mitdb')

        # Leer las anotaciones (.atr)
        annotation = wfdb.rdann(num, 'atr', pn_dir='mitdb')

        wfdb.plot_wfdb(
            record=record,
            annotation=annotation,
            title=f'MIT-BIH Arrhythmia Record {num}',
            time_units='seconds'
        )

    except Exception as e:
        print(f"No se pudo cargar el registro {num}. Error: {e}")

print("Finalizado el proceso de visualización.")

a simple vista: 112, 118, 124, 201, 205, 207, 217, 219, 220, 221, 230, 231



In [ ]:
# --- CONFIGURACIÓN ---
FS = 360
SEG_VENTANA = 5
SAMPLES_VENTANA = FS * SEG_VENTANA

# Tolerancia: cuánto se permite que varíe la media respecto a la media global
UMBRAL_TOLERANCIA = 0.25

registros_limpios = {} # Aquí van los que están rectos
registros_sucios = {}  # Aquí los que no

for num in registros:
    try:
        record = wfdb.rdrecord(num, pn_dir='mitdb')
        sig = record.p_signal[:, 0]

        # 1. Calculamos la media de TODO el registro
        media_global = np.mean(sig)

        limpios_list = []
        sucios_list = []

        total_posibles = len(sig) // SAMPLES_VENTANA

        for i in range(total_posibles):
            inicio = i * SAMPLES_VENTANA
            fin = inicio + SAMPLES_VENTANA
            trozo = sig[inicio:fin]

            # 2. Calculamos la media de este trozo de 5 segundos
            media_ventana = np.mean(trozo)

            # 3. COMPROBACIÓN: se aleja mucho de la media global del registro?
            # Si la diferencia es pequeña, asumimos que los picos y el suelo están en su sitio
            if abs(media_ventana - media_global) < UMBRAL_TOLERANCIA:
                limpios_list.append(trozo)
            else: #si la diferencia es grande es que no es ECG recto (normalmente)
                sucios_list.append(trozo)

        registros_limpios[num] = limpios_list
        registros_sucios[num] = sucios_list

        print(f"Registro {num}: {len(limpios_list)} limpios | {len(sucios_list)} sucios.")

    except Exception as e:
        print(f"Error en {num} ------------------------- !!!")

print("\nClasificación terminada.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def repasar_descartes(registros_sucios, num_registro):
    if num_registro not in registros_sucios:
        print(f"El reg {num_registro} no está en la bd")
        return

    lista_sucios = registros_sucios[num_registro]
    total_sucios = len(lista_sucios)

    if total_sucios == 0:
        print(f"El reg {num_registro} no tiene tramos sucios")
    else:
        print(f"Mostrando los primeros descartes de {num_registro} (Total: {total_sucios})")

        # Calculamos qué tan lejos está cada trozo de la media global
        # y lo comparamos con el umbral para ver el 'exceso'
        def calcular_exceso(trozo):
            diff = abs(np.mean(trozo) - media_global)
            return diff - UMBRAL_TOLERANCIA # Si es pequeño y positivo, es que casi entra

        # Ordenamos: los que tengan el exceso más pequeño (más cercanos al umbral) van primero
        tramos_frontera = sorted(lista_sucios, key=lambda x: abs(calcular_exceso(x)))

        cantidad = min(1, len(tramos_frontera))

        for i in range(cantidad):
            trozo = tramos_frontera[i]

            # --- CÁLCULO DE LA DIFERENCIA PARA ESTE TROZO ---
            media_ventana = np.mean(trozo)
            diferencia_real = abs(media_ventana - media_global)
            margen_salvacion = UMBRAL_TOLERANCIA - diferencia_real

            # AQUI IMPRIMIMOS POR CUÁNTO ESTÁ DENTRO
            print(f"-> DIF es {diferencia_real:.3f}. Se ha descartado por {margen_salvacion:.3f} mV.")

            plt.figure(figsize=(12, 4))
            plt.plot(trozo, color='red', alpha=0.7)

            # Dibujamos el cero y la media del trozo para ver por qué se fue a 'sucios'
            media_ventana = np.mean(trozo)
            plt.axhline(media_ventana, color='blue', linestyle='--', label=f'Media de este trozo: {media_ventana:.2f}')

            plt.title(f"Descarte #{i+1} del Registro {num_registro}")
            plt.xlabel("Muestras")
            plt.ylabel("Amplitud (mV)")
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.show()

In [ ]:
# --- EJECUCIÓN ---
repasar_descartes(registros_sucios, '107')
repasar_descartes(registros_sucios, '108')
repasar_descartes(registros_sucios, '111')
repasar_descartes(registros_sucios, '115')
repasar_descartes(registros_sucios, '116')
repasar_descartes(registros_sucios, '118')
repasar_descartes(registros_sucios, '121')
repasar_descartes(registros_sucios, '124')
repasar_descartes(registros_sucios, '203')
repasar_descartes(registros_sucios, '208')
repasar_descartes(registros_sucios, '213')
repasar_descartes(registros_sucios, '217')
repasar_descartes(registros_sucios, '219')
repasar_descartes(registros_sucios, '228')
repasar_descartes(registros_sucios, '233')
repasar_descartes(registros_sucios, '234')

In [ ]:
def repasar_limpios(registros_limpios, num_registro):
    if num_registro not in registros_limpios:
        print(f"El reg {num_registro} no está en la bd")
        return

    lista_limpios = registros_limpios[num_registro]
    total_limpios = len(lista_limpios)

    if total_limpios == 0:
        print(f"El reg {num_registro} no tiene tramos limpios")
    else:
        print(f"Mostrando los primeros tramos limpios de la frontera de {num_registro} (Total: {total_limpios}).")

        # Calculamos qué tan lejos está cada trozo de la media global
        # y lo comparamos con el umbral para ver el 'exceso'
        def calcular_exceso(trozo):
            diff = abs(np.mean(trozo) - media_global)
            return UMBRAL_TOLERANCIA - diff # Si es pequeño y positivo, es que casi entra

        # Ordenamos: los que tengan el exceso más pequeño (más cercanos al umbral) van primero
        tramos_frontera = sorted(lista_limpios, key=lambda x: abs(calcular_exceso(x)))

        cantidad = min(1, len(tramos_frontera))

        for i in range(cantidad):
            trozo = tramos_frontera[i]

            # --- CÁLCULO DE LA DIFERENCIA PARA ESTE TROZO ---
            media_ventana = np.mean(trozo)
            diferencia_real = abs(media_ventana - media_global)
            margen_salvacion = UMBRAL_TOLERANCIA - diferencia_real

            # AQUI IMPRIMIMOS POR CUÁNTO ESTÁ DENTRO
            print(f"-> DIF {diferencia_real:.3f}. Se ha salvado por {margen_salvacion:.3f} mV.")

            plt.figure(figsize=(12, 4))
            plt.plot(trozo, color='red', alpha=0.7)

            # Dibujamos el cero y la media del trozo para ver por qué se fue a 'sucios'
            media_ventana = np.mean(trozo)
            plt.axhline(media_ventana, color='blue', linestyle='--', label=f'Media de este trozo: {media_ventana:.2f}')

            plt.title(f"Dentro #{i+1} del Registro {num_registro}")
            plt.xlabel("Muestras")
            plt.ylabel("Amplitud (mV)")
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.show()

In [ ]:
# --- EJECUCIÓN ---
repasar_limpios(registros_limpios, '100')
repasar_limpios(registros_limpios, '101')
repasar_limpios(registros_limpios, '102')
repasar_limpios(registros_limpios, '103')
repasar_limpios(registros_limpios, '104')
repasar_limpios(registros_limpios, '105')
repasar_limpios(registros_limpios, '106')
repasar_limpios(registros_limpios, '109')
repasar_limpios(registros_limpios, '112')
repasar_limpios(registros_limpios, '113')
repasar_limpios(registros_limpios, '114')
repasar_limpios(registros_limpios, '117')
repasar_limpios(registros_limpios, '119')
repasar_limpios(registros_limpios, '122')
repasar_limpios(registros_limpios, '123')
repasar_limpios(registros_limpios, '200')
repasar_limpios(registros_limpios, '201')
repasar_limpios(registros_limpios, '202')
repasar_limpios(registros_limpios, '205')
repasar_limpios(registros_limpios, '207')

registros_limpios = [
    '100', '101', '102', '103', '104', '105', '106', '109', '113', '114',
    '122', '123', '200', '202', '205', '207', '209', '210', '214', '215',
    '220', '221', '222', '230', '231', '232'
]